In [13]:
import sys
sys.path.append("..")

In [14]:
import json

from enum import Enum
from typing import Any, TypedDict
from pydantic import BaseModel, Field, field_validator

from openai import OpenAI

In [15]:
class Intent(str, Enum):
    """
    
    """
    GENERAL = "general"
    PRODUCT_SEARCH = "product_search"
    EXACT_FACT = "exact_fact"

class IntentResult(BaseModel):
    """
    
    """
    intent: Intent
    confidence: float = Field(ge=0., le=1.)
    extracted_query: str = Field(description="Cleaned query to pass downstream")
    product_id: str | None = Field(
        default=None,
        description="Extracted product ID for EXACT_FACT queries (if present)"
    )

class Niat(TypedDict):
    """
    
    """
    GENERAL = "general"
    PRODUCT_SEARCH = "product_search"
    EXACT_FACT = "exact_fact"

In [16]:
_CLASSIFIER_PROMPT = """Kamu adalah pengklasifikasi intent untuk agen layanan pelanggan.
Klasifikasikan pesan pengguna ke salah satu dari tiga kategori:

- "general": pertanyaan umum, sapaan, atau di luar topik produk.
- "product_search": pengguna menjelajahi produk secara umum
  (berdasarkan kategori, rentang harga, atau kebutuhan).
- "exact_fact": pengguna menanyakan detail satu produk spesifik
  yang ID/namanya sudah jelas disebutkan.
"""

# Schema JSON diturunkan langsung dari model Pydantic yang sudah ada —
# satu sumber kebenaran, tidak perlu tulis ulang manual & berisiko drift.
_INTENT_JSON_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "intent_result",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "intent": {"type": "string", "enum": ["general", "product_search", "exact_fact"]},
                "confidence": {"type": "number"},
                "extracted_query": {"type": "string"},
                "product_id": {"type": ["string", "null"]},
            },
            "required": ["intent", "confidence", "extracted_query", "product_id"],
            "additionalProperties": False,
        },
    },
}


# def classify_intent(
#     client: OpenAI,
#     model: str,
#     user_message: str,
#     history: list[ChatMessage],
# ) -> IntentResult:
#     history_snippet = ""
#     if history:
#         recent = history[-3:]
#         history_snippet = "\n".join(f"{m.role.upper()}: {m.content}" for m in recent)
#         history_snippet = f"\nRecent conversation:\n{history_snippet}\n"

#     response = client.chat.completions.create(
#         model=model,
#         max_tokens=128,
#         temperature=0.0,
#         response_format=_INTENT_JSON_SCHEMA,   # <-- kunci: dipaksa di level decoding
#         messages=[
#             {"role": "system", "content": _CLASSIFIER_PROMPT},
#             {"role": "user", "content": f"{history_snippet}User message: {user_message}"},
#         ],
#     )

#     raw = response.choices[0].message.content

#     try:
#         data = json.loads(raw)
#         return IntentResult(
#             intent=Intent(data["intent"]),
#             confidence=float(data["confidence"]),
#             extracted_query=data["extracted_query"],
#             product_id=data.get("product_id"),
#         )
#     except (json.JSONDecodeError, KeyError, ValueError):
#         # Fallback tetap dipertahankan: model lama/kecil di Ollama
#         # bisa saja tidak menghormati response_format sama sekali.
#         return IntentResult(
#             intent=Intent.GENERAL, confidence=0.0,
#             extracted_query=user_message, product_id=None,
#       )

In [17]:
from langchain_ollama import ChatOllama

In [18]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by SDK, ignored by Ollama
)

In [19]:
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0,
    reasoning=False         # disable thinking mode for faster response
)

In [20]:
class Bio(TypedDict):
    nama: str
    umur: int
    alamat: str

prompt = """
extract biodata dari pesan yang masuk seperti nama, usia, dan alamat asal. Contoh:
{
    nama: andi
    alamat: jakarta
    umur: 45
}
"""

messages = [
    (
        "system",
        prompt
    ),
    ("human", "Perkenalkan nama saya doni. saya berusia 30 tahun, saya berasal dari malang"),
]

model = llm.with_structured_output(Bio)

model.invoke(messages)

{'nama': 'doni', 'umur': 30, 'alamat': 'malang'}

In [21]:
class Bio_(BaseModel):
    nama: str
    umur: int
    alamat: str

model_ = llm.with_structured_output(Bio_)

model_.invoke(messages)

Bio_(nama='doni', umur=30, alamat='malang')

In [22]:
class Emosi(BaseModel):
    emosi: str
    confidence: float

messages = [
    (
        "system",
        """
        Kamu adalah AI yang bertugas untuk mendeteksi emosi dari user yang masuk.
        Emosi yang bisa kamu deteksi adalah senang, sedih, dan marah.
        Output yang kamu hasilkan harus berbetuk json.
        Contoh:
        {
            emosi: marah,
            confidence: 0.7
        }
        confidence adalah seberapa yakin kamu dengan tebakan yang kamu berikan.
        """
    ),
    ("human", "Saku benci ya sama kamu!"),
]

model = llm.with_structured_output(Emosi)

model.invoke(messages)

Emosi(emosi='marah', confidence=0.95)

In [23]:
class Product(BaseModel):
    is_product: bool
    confidence: float
    product: str | None = Field(default=None)
    harga: int | None = Field(default=None)

messages = [
    (
        "system",
        """
        Kamu adalah AI yang bertugas untuk mendeteksi pertanyaan yang berkaitan dengan produk.
        Produk adalah barang atau benda apapun yang diperjualbelikan.
        Kamu harus mendeteksi apakah pesan user itu menanyakan tentang suatu product atau tidak.
        Jika iya, maka kamu extract nama dari product yang ditanyakan beserta harganya dalam satuan Rupiah jika ada. 
        Output yang kamu hasilkan harus berbetuk json.
        Contoh jika pertanyaan user berkaitan dengan suatu produk:
        {
            is_product: True,
            confidence: 0.7,
            product: pepsodent,
            harga: 5000
        }
        contoh jika pertanyaan user tidak berkaitan dengan suatu produk:
        {
            is_product: False,
            confidence: 0.75,
            product: None,
            harga: None
        }
        confidence adalah seberapa yakin kamu dengan tebakan yang kamu berikan.
        """
    ),
    ("human", "aku sedang bosan"),
]

model = llm.with_structured_output(Product)

model.invoke(messages)

Product(is_product=False, confidence=0.95, product=None, harga=None)

In [24]:
class Niat(str, Enum):
    GENERAL = "general"
    PRODUCT = "product"

class Product_(BaseModel):
    intent: Niat
    confidence: float
    product: str | None = Field(default=None)
    harga: int | None = Field(default=None)

messages = [
    (
        "system",
        """
        Kamu adalah AI yang bertugas untuk mendeteksi pertanyaan yang berkaitan dengan produk.
        Produk adalah barang atau benda apapun yang diperjualbelikan.
        Kamu harus mendeteksi apakah pesan user itu menanyakan tentang suatu product atau hanya pertanyaan umum.
        Jika iya, maka kamu extract nama dari product yang ditanyakan beserta harganya dalam satuan Rupiah jika ada. 
        Output yang kamu hasilkan harus berbetuk json.
        Contoh jika pertanyaan user berkaitan dengan suatu produk:
        {
            intent: product,
            confidence: 0.7,
            product: pepsodent,
            harga: 5000
        }
        contoh jika pertanyaan user tidak berkaitan dengan suatu produk:
        {
            intent: general,
            confidence: 0.75,
            product: None,
            harga: None
        }
        confidence adalah seberapa yakin kamu dengan tebakan yang kamu berikan.
        """
    ),
    ("human", ""),
]

model = llm.with_structured_output(Product_)

model.invoke(messages)

OutputParserException: Invalid json output: {
    "intent": "general",
    "confidence": 1.0,
    "product": None,
    "harga": None
}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 